In [ ]:
import tempfile
import os
import kagglehub
from pyspark.sql import SparkSession

# Choose the file inside the Kaggle dataset to load
file_path = "../datasets/2019-Oct.csv"

try:
    # Download directly into a temporary directory and avoid reusing the cache
    tmp_dir = tempfile.mkdtemp()
    dataset_dir = kagglehub.dataset_download(
        "mkechinov/ecommerce-behavior-data-from-multi-category-store",
        path=file_path,
        force_download=True,
        output_dir=tmp_dir,
    )

    if os.path.isdir(dataset_dir):
        csv_files = [f for f in os.listdir(dataset_dir) if f.endswith('.csv')]
        if not csv_files:
            raise FileNotFoundError(f"No CSV file found in downloaded dataset directory: {dataset_dir}")
        csv_path = os.path.join(dataset_dir, csv_files[0])
    else:
        csv_path = dataset_dir

    print(f"Loading CSV from: {csv_path}")

    spark = SparkSession.builder \
        .appName("KaggleEcommerceSpark") \
        .getOrCreate()

    df = spark.read.csv(
        csv_path,
        header=True,
        inferSchema=True,
        sep=",",
        encoding="ISO-8859-1",
        mode="DROPMALFORMED",
    )

    print("Schema:")
    df.printSchema()
    print("\nSample rows:")
    df.show(5, truncate=False)
    print("\nTotal rows:", df.count())

except FileNotFoundError as e:
    print(f"File not found: {e}")
except Exception as e:
    print(f"Error loading dataset: {e}")
finally:
    try:
        if 'spark' in locals():
            spark.stop()
    except Exception:
        pass


In [ ]:
# PySpark Data Exploration Guide
# This code assumes you have a DataFrame 'df' loaded from previous cells

print("=== PYSPARK DATA EXPLORATION TECHNIQUES ===\n")

# Initialize Spark if not already available
try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("DataExploration").getOrCreate()
    print("✓ SparkSession initialized")
except Exception as e:
    print(f"✗ Failed to initialize SparkSession: {e}")
    exit()

# Check if DataFrame exists
try:
    if 'df' not in locals():
        print("✗ No DataFrame 'df' found. Please load your data first.")
        exit()
    print("✓ DataFrame 'df' found")
except Exception as e:
    print(f"✗ Error accessing DataFrame: {e}")
    exit()

print()

# 1. Basic DataFrame Information
print("1. DATAFRAME BASIC INFO:")
try:
    row_count = df.count()
    col_count = len(df.columns)
    print(f"Shape: {row_count} rows x {col_count} columns")
    print(f"Columns: {df.columns}")
except Exception as e:
    print(f"✗ Error getting basic info: {e}")
print()

# 2. Schema Inspection
print("2. SCHEMA INSPECTION:")
try:
    df.printSchema()
except Exception as e:
    print(f"✗ Error printing schema: {e}")
print()

# 3. Sample Data
print("3. SAMPLE DATA:")
try:
    df.show(5, truncate=False)
except Exception as e:
    print(f"✗ Error showing sample data: {e}")
print()

# 4. Summary Statistics
print("4. SUMMARY STATISTICS:")
try:
    # For numeric columns
    numeric_cols = [col for col, dtype in df.dtypes if dtype in ['int', 'bigint', 'double', 'float', 'long']]
    if numeric_cols:
        df.select(numeric_cols).describe().show()
    else:
        print("No numeric columns found for summary statistics")
except Exception as e:
    print(f"✗ Error computing summary statistics: {e}")
print()

# 5. Data Types
print("5. COLUMN DATA TYPES:")
try:
    for col, dtype in df.dtypes:
        print(f"{col}: {dtype}")
except Exception as e:
    print(f"✗ Error getting data types: {e}")
print()

# 6. Null Values Check
print("6. NULL VALUES ANALYSIS:")
try:
    from pyspark.sql.functions import col, sum as spark_sum
    null_counts = df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
    null_counts.show()
except Exception as e:
    print(f"✗ Error analyzing null values: {e}")
print()

# 7. Unique Values Count (sample for performance)
print("7. UNIQUE VALUES COUNT (first 5 columns):")
try:
    for col in df.columns[:5]:  # Limit to first 5 columns for performance
        try:
            unique_count = df.select(col).distinct().count()
            print(f"{col}: {unique_count} unique values")
        except Exception as col_error:
            print(f"{col}: Error counting unique values - {col_error}")
except Exception as e:
    print(f"✗ Error counting unique values: {e}")
print()

# 8. Value Distributions (for categorical columns)
print("8. VALUE DISTRIBUTIONS (Top 5 for first 2 string columns):")
try:
    string_cols = [col for col, dtype in df.dtypes if dtype == 'string']
    for col in string_cols[:2]:  # Limit to first 2 string columns
        print(f"\nTop 5 values in '{col}':")
        try:
            df.groupBy(col).count().orderBy("count", ascending=False).show(5, truncate=False)
        except Exception as col_error:
            print(f"Error getting distribution for {col}: {col_error}")
except Exception as e:
    print(f"✗ Error analyzing value distributions: {e}")
print()

print("=== EXPLORATION COMPLETE ===")
print("\n💡 Tips for PySpark Data Exploration:")
print("• Use df.cache() for frequently accessed DataFrames")
print("• Use df.persist() for custom storage levels")
print("• Use df.sample() for large datasets")
print("• Use df.filter() before aggregations for better performance")
print("• Use df.explain() to understand query execution plans")

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.